# Convert the Signtext SavedModel to TensorFlow.js

This notebook runs **in Google Colab**, on Google's machine. Nothing is installed on your own computer.

## What it does

1. Builds an **isolated Python 3.11 environment inside Colab** with `uv`, pinned to the exact versions that trained and converted the model: TensorFlow 2.21.0, NumPy 2.4.6, tf-keras 2.21.0, TensorFlow.js 4.22.0. Colab's own Python is left alone, so its ever-changing packaged versions cannot shift the pins.
2. Uploads a zip of the SavedModel folder.
3. Pre-flight checks: the zip landed on disk, it contains `saved_model.pb` **and** a `variables/` folder, and the isolated environment can import the converter.
4. Converts from `tf_saved_model` to a TensorFlow.js **Graph Model**, then post-flight checks the result: `model.json` plus a weight shard exist, the converted input width is 126, the converted output width is 28, and the model still turns 5 random inputs into rows of 28 probabilities that each add up to about 1.
5. Zips the result with `model.json` and the weight shard **at the top level of the zip**, prints the names and sizes, and starts a download.

The model was trained and saved with TensorFlow **2.21.0** and the converter inside the isolated environment is also 2.21.0, so there is no save/convert version gap.

## Before you start

Zip the folder `ml/checkpoints/alphabet_savedmodel` on your computer. The zip **must contain the folder itself**, so that `saved_model.pb` sits one level down inside it. Colab can only upload files, not folders, which is why the zip step comes first.

Run the cells from top to bottom with **Shift+Enter**.

## Step 1 - Build the isolated conversion environment

This installs `uv`, creates a Python 3.11 venv, and installs two sets of pins:

- **With dependencies:** `tensorflow==2.21.0`, `numpy==2.4.6`, `tf-keras==2.21.0`, `six==1.17.0`, `h5py==3.14.0`, `packaging==26.3`, `importlib-resources==7.1.0`.
- **Without dependencies** (`--no-deps`): `tensorflowjs==4.22.0`, `jax==0.4.35`, `jaxlib==0.4.35`, `ml-dtypes==0.6.0`. `--no-deps` stops pip from re-resolving these against newer releases.

`tensorflow_hub` and `tensorflow-decision-forests` are **not** installed on purpose (they force older pinned packages, and one of them cannot even load against TF 2.21). The converter imports both at startup without ever calling them for a plain Keras graph; Step 4's launcher stubs them out and raises a loud error if a conversion ever really touches one.

This cell may take several minutes the first time (TensorFlow is a large download). It only writes into `signtext_convert/venv` next to the notebook folder.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

BASE = Path(os.environ.get("SIGNTEXT_CONVERT_BASE", str(Path.cwd() / "signtext_convert")))
BASE.mkdir(parents=True, exist_ok=True)
venv_dir = BASE / "venv"

# uv is a single small binary and Colab does not have it, so pip it into
# Colab's Python first. This does not touch the venv built next.
print("Installing uv into Colab's Python...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)

uv = shutil.which("uv")
if not uv:
    uv = str(Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"))
if not Path(uv).exists():
    raise RuntimeError("uv was installed but not found on PATH. Rerun this cell.")

print("Creating the Python 3.11 venv...")
venv_py = venv_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if venv_py.is_file():
    print("A venv already exists; reusing it.")
else:
    # A fresh Colab VM starts empty, so this normally creates the venv. Reruns
    # of this cell inside the same VM reuse what is already there.
    if venv_dir.exists() and not venv_py.exists():
        shutil.rmtree(venv_dir)
    r = subprocess.run([uv, "venv", "--python", "3.11", str(venv_dir)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "uv could not create the venv.\n"
            f"--- stdout ---\n{r.stdout}\n--- stderr ---\n{r.stderr}\n"
            "Rerun this cell."
        )
    if r.stdout.strip():
        print(r.stdout.strip())

print("Installing pinned packages (this downloads TensorFlow, a few minutes)...")

(BASE / "requirements-pinned.txt").write_text(
    "tensorflow==2.21.0\n"
    "numpy==2.4.6\n"
    "tf-keras==2.21.0\n"
    "six==1.17.0\n"
    "h5py==3.14.0\n"
    "packaging==26.3\n"
    "importlib-resources==7.1.0\n",
    encoding="utf-8",
)
(BASE / "requirements-nodeps.txt").write_text(
    "tensorflowjs==4.22.0\n"
    "jax==0.4.35\n"
    "jaxlib==0.4.35\n"
    "ml-dtypes==0.6.0\n",
    encoding="utf-8",
)

for req, no_deps in (("requirements-pinned.txt", False), ("requirements-nodeps.txt", True)):
    cmd = [uv, "pip", "install", "--python", str(venv_py)]
    if no_deps:
        cmd.append("--no-deps")
    cmd += ["--requirement", str(BASE / req)]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(
            f"uv pip install {req} failed with exit code {result.returncode}. "
            "Rerun this cell."
        )

print("The isolated environment is ready. It is only used for this conversion.")

## Step 2 - Upload the zip

Choose the zip you made from `ml/checkpoints/alphabet_savedmodel`.

The cell first deletes the scratch folder and any zip left over from an earlier run, then uploads, then reports the file that actually landed on disk. A successful upload prints the byte size and the line `Using <file>`.

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import files

cwd = Path.cwd()
BASE = Path(os.environ.get("SIGNTEXT_CONVERT_BASE", str(cwd / "signtext_convert")))
work = BASE / "scratch"

# Clear leftovers from an earlier run. The venv from Step 1 lives beside the
# scratch folder (under BASE) and is deliberately NOT deleted here.
cleared = []
if work.exists():
    shutil.rmtree(work)
    cleared.append("scratch/")
for stale_zip in sorted(cwd.glob("*.zip")):
    stale_zip.unlink()
    cleared.append(stale_zip.name)
if cleared:
    print("Cleared from an earlier run:", ", ".join(cleared))

print("Select the zip of alphabet_savedmodel...")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was selected. Run this cell again and pick the zip.")

# files.upload() returns {name: bytes}. The value is the file contents, not a
# size, so the length has to be taken with len().
zip_name = next(iter(uploaded))
data = uploaded[zip_name]
if not isinstance(data, (bytes, bytearray)):
    raise RuntimeError(
        f"files.upload() returned {type(data).__name__} for {zip_name!r}, expected bytes."
    )
print(f"Uploaded {zip_name} ({len(data):,} bytes).")

if not zip_name.lower().endswith(".zip"):
    raise RuntimeError(
        f"Expected a .zip file but got {zip_name!r}. "
        "Zip the alphabet_savedmodel folder first, then upload again."
    )

# Trust the file on disk rather than the uploaded name. Colab can save an
# upload under a different name (for example "alphabet_savedmodel (2).zip"),
# and building a path from zip_name would then open a leftover file.
written = [p for p in sorted(cwd.glob("*.zip")) if p.stat().st_size == len(data)]
if not written:
    raise RuntimeError(
        f"Upload reported {zip_name!r} ({len(data):,} bytes) but no matching "
        f".zip was written to {cwd}."
    )
if len(written) > 1:
    raise RuntimeError(
        "More than one .zip in " + str(cwd) + " matches that size: "
        + ", ".join(p.name for p in written)
        + ". Run this cell again; it clears old zips first."
    )

zip_path = written[0]
print(f"Using {zip_path.name} ({zip_path.stat().st_size:,} bytes)")

## Step 3 - Unpack and find the SavedModel

Unpacks the zip and looks for the one folder that holds `saved_model.pb` together with a `variables/` subfolder.

In [ ]:
import os
import shutil
from pathlib import Path

cwd = Path.cwd()
BASE = Path(os.environ.get("SIGNTEXT_CONVERT_BASE", str(cwd / "signtext_convert")))
work = BASE / "scratch"

if "zip_path" not in globals() or not Path(zip_path).is_file():
    raise RuntimeError("zip_path is missing. Run Step 2 again before this cell.")

if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

print("Unpacking", Path(zip_path).name)
shutil.unpack_archive(str(zip_path), str(work), format="zip")
print("Unpacked to", work)

candidates = sorted({p.parent for p in work.rglob("saved_model.pb")})
if not candidates:
    raise RuntimeError(
        "No saved_model.pb found in the zip. "
        "Zip the alphabet_savedmodel folder itself, not its parent folder."
    )
if len(candidates) > 1:
    raise RuntimeError(
        "The zip holds more than one SavedModel:\n"
        + "\n".join(f"  - {p}" for p in candidates)
        + "\nZip only the alphabet_savedmodel folder."
    )

saved_model_dir = candidates[0]
if not (saved_model_dir / "variables").is_dir():
    raise RuntimeError(
        f"{saved_model_dir} is missing its variables/ folder. This does not look "
        "like a complete SavedModel. Re-zip the alphabet_savedmodel folder and "
        "run Steps 2 and 3 again."
    )

print("SavedModel folder:", saved_model_dir)
for child in sorted(saved_model_dir.iterdir()):
    print("  ", child.name)

## Step 4 - Convert and verify

First three pre-flight checks (rerun guidance in the messages), then the conversion, then two post-flight checks:

1. The converted folder holds `model.json` plus at least one `.bin` weight shard, its format is `graph-model`, and `model.json` reports an input width of 126 and an output width of 28.
2. The same isolated environment loads the SavedModel and confirms 5 random inputs each produce 28 probabilities that add up to about 1.

Load the produced folder later with `tf.loadGraphModel()`.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

BASE = Path(os.environ.get("SIGNTEXT_CONVERT_BASE", str(Path.cwd() / "signtext_convert")))
venv_dir = BASE / "venv"
work = BASE / "scratch"
venv_py = venv_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

# --- Pre-flight 1: the uploaded zip from Step 2 ---
if "zip_path" not in globals() or not Path(zip_path).is_file():
    raise RuntimeError("The uploaded zip is missing. Run Step 2 (upload), then this cell.")

# --- Pre-flight 2: the unpacked SavedModel from Step 3 ---
if "saved_model_dir" not in globals():
    raise RuntimeError("The SavedModel folder is missing. Run Step 2, then Step 3, then this cell.")
saved_model_dir = Path(saved_model_dir)
if not (saved_model_dir / "saved_model.pb").is_file() or not (saved_model_dir / "variables").is_dir():
    raise RuntimeError(
        f"{saved_model_dir} does not contain saved_model.pb plus a variables/ folder. "
        "Re-run Steps 2-3, then this cell."
    )

# The converter launcher. tensorflowjs imports tensorflow_decision_forests and
# tensorflow_hub at import time even though a plain Keras SavedModel never uses
# them. Installing those two would drag in packages that change the pinned
# TensorFlow, so they are NOT installed; the launcher stubs them out.
launcher = work / "run_converter.py"
launcher.write_text(
'''"""Run the TensorFlow.js converter inside the isolated Python 3.11 environment."""

import sys

import numpy as np

# NumPy 2 removed a few aliases that tensorflowjs reads during startup.
# Restore the harmless ones before tensorflowjs is imported.
for _alias, _value in (("object", object), ("bool", bool), ("int", int), ("float", float)):
    if not hasattr(np, _alias):
        setattr(np, _alias, _value)

def _missing_module(name):
    class _Missing:
        def __getattr__(_ignored_self, attr):
            raise RuntimeError(
                "Conversion tried to use " + repr(name) + " (attribute " + repr(attr) + "), "
                "which is not installed in the isolated environment. That only happens "
                "for TF-DF or TF-Hub SavedModels; this model is a plain Keras graph. "
                "If it unexpectedly is not, install the package and rerun Step 4."
            )
    return _Missing()

for _required_optional in ("tensorflow_decision_forests", "tensorflow_hub"):
    try:
        __import__(_required_optional)
    except ImportError:
        sys.modules[_required_optional] = _missing_module(_required_optional)

import tensorflow as tf
import tf_keras
import tensorflowjs.converters.converter as converter
from tensorflowjs import version as tfjs_version

if "--check" in sys.argv:
    print("isolated environment check passed")
    print("  Python       ", sys.version.split()[0])
    print("  NumPy        ", np.__version__)
    print("  TensorFlow   ", tf.__version__)
    print("  tf-keras     ", tf_keras.__version__)
    print("  tensorflowjs ", tfjs_version.version)
    sys.exit(0)

converter.convert([arg for arg in sys.argv[1:] if arg != "--check"])
''',
    encoding="utf-8",
)

# A second script does the prediction check in the same isolated env.
predict_check = work / "predict_check.py"
predict_check.write_text(
'''"""Sanity-check the exported SavedModel from inside the isolated environment."""

import sys

import numpy as np
import tensorflow as tf

saved_model_dir = sys.argv[1]
model = tf.saved_model.load(saved_model_dir)
signature = model.signatures["serving_default"]

args, kwargs = signature.structured_input_signature
if len(kwargs) != 1:
    raise RuntimeError(f"Expected exactly one model input, got {len(kwargs)}.")
input_name, input_spec = next(iter(kwargs.items()))
input_shape = input_spec.shape.as_list()
if input_shape[1:] != [126]:
    raise RuntimeError(f"Expected model input shape [None, 126], got {input_shape}.")
print(f"Input:  {input_name}, shape {input_shape}, dtype {input_spec.dtype.name}")

outputs = signature.structured_outputs
if len(outputs) != 1:
    raise RuntimeError(f"Expected exactly one model output, got {len(outputs)}.")
output_name = next(iter(outputs))
print(f"Output: {output_name}, shape [None, 28]")

rng = np.random.default_rng(0)
X = rng.normal(size=(5, 126)).astype("float32")
raw = signature(tf.constant(X))[output_name]
probs = raw.numpy()
if probs.shape != (5, 28):
    raise RuntimeError(f"Expected prediction shape (5, 28), got {probs.shape}.")
sums = probs.sum(axis=1)
for i, s in enumerate(sums):
    print(f"  Prediction {i}: 28 probabilities, row sum {s:.6f}")
if not np.allclose(sums, 1.0, atol=1e-3):
    raise RuntimeError(f"Softmax row sums should be close to 1, got {sums}.")
print("Prediction check passed: 5 x 28 probabilities, each row sums to ~1.")
''',
    encoding="utf-8",
)

# --- Pre-flight 3: the isolated environment imports the converter ---
if not venv_py.is_file():
    raise RuntimeError(
        f"The isolated environment is missing ({venv_dir}). "
        "Run Step 1 to build it, then this cell."
    )

print("Checking the isolated environment...")
r = subprocess.run([str(venv_py), str(launcher), "--check"], capture_output=True, text=True)
if r.stdout.strip():
    print(r.stdout.strip())
if r.returncode != 0:
    raise RuntimeError(
        "The isolated environment could not import the converter.\n"
        f"--- stdout ---\n{r.stdout}\n--- stderr ---\n{r.stderr}\n"
        "Rerun Step 1 to rebuild it, then this cell."
    )

# --- Convert ---
tfjs_dir = work / "tfjs_model"
if tfjs_dir.exists():
    shutil.rmtree(tfjs_dir)

command = [
    str(venv_py), str(launcher),
    "--input_format=tf_saved_model",
    "--output_format=tfjs_graph_model",
    "--signature_name=serving_default",
    "--saved_model_tags=serve",
    str(saved_model_dir),
    str(tfjs_dir),
]
print("Running:", " ".join(command))
result = subprocess.run(command, capture_output=True, text=True)
if result.stdout.strip():
    print(result.stdout.strip())
if result.returncode != 0:
    raise RuntimeError(
        "tensorflowjs_converter failed with exit code " + str(result.returncode) + ".\n"
        f"--- stdout ---\n{result.stdout}\n--- stderr ---\n{result.stderr}\n"
        "This is not an upload problem, so Steps 2-3 are fine. Rerun this cell."
    )
if result.stderr.strip():
    print(result.stderr.strip())

# --- Post-flight 1: the converted artifact exists and has the right shape ---
model_json_file = tfjs_dir / "model.json"
bin_files = sorted(tfjs_dir.glob("*.bin"))
if not model_json_file.is_file() or not bin_files:
    raise RuntimeError(
        "Conversion finished but the output has no model.json or .bin file: "
        + ", ".join(p.name for p in sorted(tfjs_dir.iterdir()))
        + ". Rerun this cell."
    )
data = json.loads(model_json_file.read_text(encoding="utf-8"))
if data.get("format") != "graph-model":
    raise RuntimeError(f"Expected format graph-model, got {data.get('format')!r}.")


def last_dim(entry):
    dims = (entry.get("tensorShape") or {}).get("dim") or []
    return dims[-1].get("size") if dims else None


for name, entry in (data.get("signature", {}).get("inputs") or {}).items():
    if last_dim(entry) not in ("126", 126):
        raise RuntimeError(
            f"Converted input {name!r} has width {last_dim(entry)}, expected 126."
        )
for name, entry in (data.get("signature", {}).get("outputs") or {}).items():
    if last_dim(entry) != "28":
        raise RuntimeError(
            f"Converted output {name!r} has width {last_dim(entry)}, expected 28."
        )
print("Converted artifact: input width 126, output width 28 (read from model.json).")

# --- Post-flight 2: the model still predicts a valid distribution ---
print("Running prediction check on the SavedModel...")
r = subprocess.run([str(venv_py), str(predict_check), str(saved_model_dir)], capture_output=True, text=True)
if r.stdout.strip():
    print(r.stdout.strip())
if r.returncode != 0:
    raise RuntimeError(
        "The prediction check failed.\n"
        f"--- stdout ---\n{r.stdout}\n--- stderr ---\n{r.stderr}\n"
        "Conversion produced files, but the underlying model did not pass. Rerun this cell."
    )

print("Converted. Output folder:", tfjs_dir)
for child in sorted(tfjs_dir.iterdir()):
    print(f"  {child.name} ({child.stat().st_size:,} bytes)")


## Step 5 - Download the result

Zips the TensorFlow.js folder so `model.json` and the weight shard end up **at the top level of the zip**, prints every entry name and size, then starts a download.

Unzip it into `frontend/public/models/sign-model/` so the web app can load it.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import files

BASE = Path(os.environ.get("SIGNTEXT_CONVERT_BASE", str(Path.cwd() / "signtext_convert")))
work = BASE / "scratch"
tfjs_dir = work / "tfjs_model"

if not (tfjs_dir / "model.json").is_file():
    raise RuntimeError(
        f"No model.json in {tfjs_dir}. Convert first (Step 4), then run this cell."
    )

# The zip target lives OUTSIDE tfjs_dir (work/tfjs_model.zip). Writing it inside
# tfjs_dir would make the archiver zip the growing zip file into itself.
archive = shutil.make_archive(str(work / "tfjs_model"), "zip", root_dir=str(tfjs_dir))
print("Zipped to", archive)

print("Contents:")
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        print(f"  {info.filename} ({info.file_size:,} bytes)")
print(f"Total: {Path(archive).stat().st_size:,} bytes")

files.download(archive)
print("Download started. If your browser blocked it, allow downloads for this site.")

## Done

You should now have `tfjs_model.zip`. Unzip it into `frontend/public/models/sign-model/` next to `class_names.json`. The folder should contain `model.json` and the weight shard file(s) at the top level.

## Notes on what was tested

The pinned environment, the conversion, and both post-flight checks were run against the real `alphabet_savedmodel` before this notebook was finished, using the same `uv` workflow and the same pins: TensorFlow 2.21.0, NumPy 2.4.6, tf-keras 2.21.0, TensorFlow.js 4.22.0, jax/jaxlib 0.4.35, ml-dtypes 0.6.0. The two Colab-specific pieces - `files.upload()` and `files.download()` - and uv downloading a standalone CPython 3.11 inside Colab could not be exercised off Colab. On your first Colab run, compare the version lines printed by the Step 4 environment check against the list above.